# Tunable compliance — emergent grasps

## In-hand manipulation — stiffness shift

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, sys
from pathlib import Path

sys.path.insert(0, os.path.join('../..'))
plt.style.use(os.path.join('../..', 'plot_config.mplstyle'))

FINGERTIPS = ['thumb', 'index', 'middle', 'ring', 'pinky']
PHASES     = ['uniform', 'asym_a', 'asym_b']
PHASE_LABELS = {'uniform': 'Uniform', 'asym_a': 'Asym A', 'asym_b': 'Asym B'}

OUTPUT_INHAND = os.path.join('outputs', 'inhand_manipulation')

def load_latest_inhand():
    folder = Path(OUTPUT_INHAND)
    if not folder.exists():
        return None
    files = sorted(folder.glob('inhand_run_*.csv'))
    return pd.read_csv(files[-1]) if files else None

df_inhand = load_latest_inhand()

In [ ]:
fig, ax = plt.subplots()

colors = ['#56B4E9', '#E69F00', '#D55E00']
x = np.arange(len(FINGERTIPS))
width = 0.25

for i, (phase, color) in enumerate(zip(PHASES, colors)):
    means = []
    for f in FINGERTIPS:
        if df_inhand is not None:
            rows = df_inhand[df_inhand['phase'] == phase]
            disp_mag = np.sqrt(
                rows[f'disp_{f}_x_m']**2 +
                rows[f'disp_{f}_y_m']**2 +
                rows[f'disp_{f}_z_m']**2
            )
            means.append(float(disp_mag.mean()) * 1e3)
        else:
            means.append(0.0)
    ax.bar(x + i * width, means, width, label=PHASE_LABELS[phase], color=color)

ax.set_xticks(x + width)
ax.set_xticklabels(FINGERTIPS)
ax.set_xlabel('Finger')
ax.set_ylabel('Mean tip displacement [mm]')
ax.legend()
fig.tight_layout()
os.makedirs(OUTPUT_INHAND, exist_ok=True)
fig.savefig(os.path.join(OUTPUT_INHAND, 'inhand_tip_displacement.pdf'), bbox_inches='tight')
plt.show()

## Dynamic grasping — soft vs stiff vs adaptive

In [ ]:
OUTPUT_GRASP = os.path.join('outputs', 'dynamic_grasp')
CONDITIONS   = ['soft', 'stiff', 'adaptive']
COLORS_DG    = {'soft': '#56B4E9', 'stiff': '#D55E00', 'adaptive': '#009E73'}

def load_dynamic(condition):
    folder = Path(OUTPUT_GRASP)
    if not folder.exists():
        return None
    files = sorted(folder.glob(f'dynamic_grasp_{condition}_*.csv'))
    return pd.read_csv(files[-1]) if files else None

N_GRID_DG = 300

def interp_trace(arr, n=N_GRID_DG):
    x = np.linspace(0, 1, len(arr))
    return np.interp(np.linspace(0, 1, n), x, arr)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))

for cond, color in COLORS_DG.items():
    df = load_dynamic(cond)
    if df is None or df.empty:
        continue
    closed = df[df['phase'] != 'open']
    if closed.empty:
        continue
    T = np.linspace(0, len(closed) / 30, N_GRID_DG)  # ~30 Hz logging

    # Index MCP angle (motor 7)
    axes[0].plot(T, np.rad2deg(interp_trace(closed['q_7'].to_numpy())),
                 color=color, label=cond.capitalize())
    # Index MCP torque (motor 7)
    axes[1].plot(T, interp_trace(closed['tau_7'].to_numpy()),
                 color=color, label=cond.capitalize())

axes[0].set_xlabel('Time after close [s]')
axes[0].set_ylabel('Index MCP angle [deg]')
axes[0].legend()
axes[1].set_xlabel('Time after close [s]')
axes[1].set_ylabel('Index MCP torque [N·m]')
axes[1].legend()
fig.tight_layout()
os.makedirs(OUTPUT_GRASP, exist_ok=True)
fig.savefig(os.path.join(OUTPUT_GRASP, 'dynamic_grasp_comparison.pdf'), bbox_inches='tight')
plt.show()